multi head attention with grouped query atention 


In [ ]:
from MLconcepts import attention
class MHA(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dropout = 0.3, qkv_bias = False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads",
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.wq = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.wk = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.wv = nn.Linear(d_in,d_out, bias = qkv_bias)
        self.out = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length,context_length), diagonal = -1)
        )

    def forward(self, x):
        B, nums_tokens, d_in = x.shape
        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        Q = self.wq(x).view(B,nums_tokens,self.num_heads, self.head_dim )
        K = self.wk(x).view(B,nums_tokens,self.num_heads, self.head_dim )
        V = self.wv(x).view(B,nums_tokens,self.num_heads, self.head_dim )
        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)  num_tokens, head_dim)
        Q = Q.transpose(1,2)    
        K = K.transpose(1,2)
        V = V.transpose(1,2) 

        attn_scores = Q @ K.transpose(2,3)  
        
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -float('inf'))
        attn_weight = torch.softmax(attn_scores / math.sqrt(self.d_out), dim = 1)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [8]:
import torch 

a = torch.randn(1,2,3)
print(a.shape)
b = a.transpose(1,2)
print(b.shape)

torch.Size([1, 2, 3])
torch.Size([1, 3, 2])
